# Partial OT Active Mass Selection

This notebook generates `fig:partial-ot-active-mass`.  It solves the fixed-mass partial optimal transport problem
$$
\operatorname{POT}_m(a,b)
=\min_{P\geq0}\{\langle C,P\rangle:
P\mathbf 1\leq a,
P^\top\mathbf 1\leq b,
\mathbf 1^\top P\mathbf 1=m\}.
$$
The same one-dimensional source and target histograms are used in all panels.  The source is a two-Gaussian mixture located on the left, while the target is a two-Gaussian mixture located on the right.  As the prescribed transported mass \(m\) decreases, the optimizer keeps only the cheapest active source and target submarginals and leaves the rest unmatched.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")

ROOT = Path.cwd()
if (ROOT / "notebooks-figures" / "figure_style.py").exists():
    FIGROOT = ROOT / "notebooks-figures"
elif (ROOT / "figure_style.py").exists():
    FIGROOT = ROOT
    ROOT = FIGROOT.parent
elif (ROOT.parent / "notebooks-figures" / "figure_style.py").exists():
    ROOT = ROOT.parent
    FIGROOT = ROOT / "notebooks-figures"
else:
    raise RuntimeError("Could not locate notebooks-figures/figure_style.py")

sys.path.insert(0, str(FIGROOT))

import matplotlib.pyplot as plt
import numpy as np
import ot
from matplotlib.colors import LinearSegmentedColormap

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    AXIS_LINE_WIDTH,
    figure_dir,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

NAME = "partial-ot-active-mass"
OUT = figure_dir(NAME)
THUMB_DIR = FIGROOT / "thumbnails"
THUMB_DIR.mkdir(exist_ok=True)
ARXIV_OUT = ROOT / "arxiv" / "figures"
ARXIV_OUT.mkdir(parents=True, exist_ok=True)


## Two shifted Gaussian mixtures

The example uses two Gaussian components for each marginal.  The source mixture is concentrated on the left, whereas the target mixture is concentrated on the right.  This makes partial OT select the closest portions first and progressively discard mass that would require longer transport.

In [ ]:
def gaussian(x, mean, sigma):
    """Unnormalized one-dimensional Gaussian density."""
    z = (x - mean) / sigma
    return np.exp(-0.5 * z**2) / sigma


def normalize(w):
    w = np.maximum(np.asarray(w, dtype=float), 0.0)
    return w / w.sum()


grid = np.linspace(-3.15, 3.15, 190)
dx = grid[1] - grid[0]

# Two-component mixtures: red is globally shifted left, blue shifted right.
alpha_shape = (
    0.56 * gaussian(grid, -2.08, 0.34)
    + 0.44 * gaussian(grid, -0.76, 0.24)
)
beta_shape = (
    0.43 * gaussian(grid, 0.64, 0.26)
    + 0.57 * gaussian(grid, 1.93, 0.36)
)

alpha = normalize(alpha_shape)
beta = normalize(beta_shape)
alpha_density = alpha / dx
beta_density = beta / dx

cost = (grid[:, None] - grid[None, :]) ** 2
MASS_VALUES = [0.90, 0.65, 0.42, 0.22]
PANEL_NAMES = ["mass-090", "mass-065", "mass-042", "mass-022"]

plans = []
for mass in MASS_VALUES:
    P = ot.partial.partial_wasserstein(alpha, beta, cost, m=mass, log=False)
    plans.append(P)
    assert np.isclose(P.sum(), mass, atol=1e-10)
    assert np.all(P.sum(axis=1) <= alpha + 1e-10)
    assert np.all(P.sum(axis=0) <= beta + 1e-10)

print("transported masses:", [float(P.sum()) for P in plans])


## Matrix panels with active marginals

The central image is the exact discrete partial plan, rendered with a power-law contrast in each panel so that the low-mass plans remain readable.  The side plots compare the prescribed marginals with the active submarginals selected by the plan.

In [ ]:
plan_cmap = LinearSegmentedColormap.from_list(
    "partial_plan",
    ["#ffffff", "#f6edf7", "#d8b5de", VIOLET],
)

all_row_densities = [P.sum(axis=1) / dx for P in plans]
all_col_densities = [P.sum(axis=0) / dx for P in plans]
side_scale = 1.10 * max(
    alpha_density.max(),
    beta_density.max(),
    max(row.max() for row in all_row_densities),
    max(col.max() for col in all_col_densities),
)


xlim = (grid[0], grid[-1])


def remove_side_axes(ax):
    remove_axes(ax)
    ax.set_facecolor("white")


def box_matrix_axis(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(AXIS_LINE_WIDTH)
        spine.set_edgecolor("#2f2f2f")


def plan_image(P):
    # Per-panel normalization keeps small transported masses visible.
    return (P / max(P.max(), 1e-15)) ** 0.34


def draw_panel(P, path):
    row_density = P.sum(axis=1) / dx
    col_density = P.sum(axis=0) / dx

    fig = plt.figure(figsize=(2.42, 2.20))
    gs = fig.add_gridspec(
        2,
        2,
        width_ratios=[0.31, 1.0],
        height_ratios=[0.31, 1.0],
        wspace=0.026,
        hspace=0.026,
    )
    ax_corner = fig.add_subplot(gs[0, 0])
    ax_top = fig.add_subplot(gs[0, 1])
    ax_left = fig.add_subplot(gs[1, 0])
    ax = fig.add_subplot(gs[1, 1])

    remove_side_axes(ax_corner)
    remove_side_axes(ax_top)
    remove_side_axes(ax_left)

    ax.imshow(
        plan_image(P),
        origin="lower",
        extent=[grid[0], grid[-1], grid[0], grid[-1]],
        cmap=plan_cmap,
        interpolation="bilinear",
        aspect="auto",
        vmin=0.0,
        vmax=1.0,
    )
    ax.set_xlim(*xlim)
    ax.set_ylim(*xlim)
    box_matrix_axis(ax)

    ax_left.fill_betweenx(grid, 0, alpha_density, color=RED, alpha=0.08, linewidth=0)
    ax_left.fill_betweenx(grid, 0, row_density, color=VIOLET, alpha=0.16, linewidth=0)
    ax_left.plot(alpha_density, grid, color=RED, lw=0.78, alpha=0.42)
    ax_left.plot(row_density, grid, color=VIOLET, lw=1.18)
    ax_left.set_ylim(*xlim)
    ax_left.set_xlim(side_scale, 0.0)

    ax_top.fill_between(grid, 0, beta_density, color=BLUE, alpha=0.08, linewidth=0)
    ax_top.fill_between(grid, 0, col_density, color=VIOLET, alpha=0.16, linewidth=0)
    ax_top.plot(grid, beta_density, color=BLUE, lw=0.78, alpha=0.42)
    ax_top.plot(grid, col_density, color=VIOLET, lw=1.18)
    ax_top.set_xlim(*xlim)
    ax_top.set_ylim(0.0, side_scale)

    save_pdf(fig, path, pad_inches=0.01)
    plt.close(fig)


for P, name in zip(plans, PANEL_NAMES):
    draw_panel(P, OUT / f"{name}.pdf")
    shutil.copy2(OUT / f"{name}.pdf", ARXIV_OUT / f"{NAME}--{name}.pdf")

print(f"Exported {len(PANEL_NAMES)} PDF panels to {OUT}")

In [ ]:
def draw_panel_on_grid(fig, subspec, P, mass):
    row_density = P.sum(axis=1) / dx
    col_density = P.sum(axis=0) / dx
    gs = subspec.subgridspec(
        2,
        2,
        width_ratios=[0.31, 1.0],
        height_ratios=[0.31, 1.0],
        wspace=0.026,
        hspace=0.026,
    )
    ax_corner = fig.add_subplot(gs[0, 0])
    ax_top = fig.add_subplot(gs[0, 1])
    ax_left = fig.add_subplot(gs[1, 0])
    ax = fig.add_subplot(gs[1, 1])
    remove_side_axes(ax_corner)
    remove_side_axes(ax_top)
    remove_side_axes(ax_left)

    ax.imshow(
        plan_image(P),
        origin="lower",
        extent=[grid[0], grid[-1], grid[0], grid[-1]],
        cmap=plan_cmap,
        interpolation="bilinear",
        aspect="auto",
        vmin=0.0,
        vmax=1.0,
    )
    ax.set_xlim(*xlim)
    ax.set_ylim(*xlim)
    box_matrix_axis(ax)

    ax_left.fill_betweenx(grid, 0, alpha_density, color=RED, alpha=0.08, linewidth=0)
    ax_left.fill_betweenx(grid, 0, row_density, color=VIOLET, alpha=0.16, linewidth=0)
    ax_left.plot(alpha_density, grid, color=RED, lw=0.70, alpha=0.42)
    ax_left.plot(row_density, grid, color=VIOLET, lw=1.02)
    ax_left.set_ylim(*xlim)
    ax_left.set_xlim(side_scale, 0.0)

    ax_top.fill_between(grid, 0, beta_density, color=BLUE, alpha=0.08, linewidth=0)
    ax_top.fill_between(grid, 0, col_density, color=VIOLET, alpha=0.16, linewidth=0)
    ax_top.plot(grid, beta_density, color=BLUE, lw=0.70, alpha=0.42)
    ax_top.plot(grid, col_density, color=VIOLET, lw=1.02)
    ax_top.set_xlim(*xlim)
    ax_top.set_ylim(0.0, side_scale)
    ax_top.set_title(rf"$m={mass:.2f}$", pad=1.0, fontsize=10)


def make_thumbnail(path):
    fig = plt.figure(figsize=(8.9, 2.15))
    outer = fig.add_gridspec(1, 4, left=0.01, right=0.99, bottom=0.02, top=0.88, wspace=0.17)
    for k, (P, mass) in enumerate(zip(plans, MASS_VALUES)):
        draw_panel_on_grid(fig, outer[k], P, mass)
    path.parent.mkdir(exist_ok=True)
    fig.savefig(path, dpi=185, bbox_inches="tight", pad_inches=0.025)
    plt.close(fig)


make_thumbnail(THUMB_DIR / f"{NAME}.png")
print(f"Thumbnail: {THUMB_DIR / f'{NAME}.png'}")

## Figure preview

The manuscript arranges the exported PDF panels directly and adds the transported-mass labels in LaTeX. The thumbnail below previews the four-panel composition used in the figure notebook gallery.

![Partial OT active mass selection](thumbnails/partial-ot-active-mass.png)